# ChainValley — offline analysis (§4.4)

Load `artifacts/runs/*.json`, compute survival rounds, Gini on cumulative harvest, violation rate, and Mann–Whitney U (soft vs hard, n=10 each).

In [ ]:
from pathlib import Path
import json

import numpy as np
from scipy.stats import mannwhitneyu

runs_dir = Path("..") / "artifacts" / "runs"
files = sorted(runs_dir.glob("*.json")) if runs_dir.exists() else []
records = []
for p in files:
    with open(p, encoding="utf-8") as f:
        records.append(json.load(f))
len(records), [r.get("condition") for r in records[:4]]

In [ ]:
def gini(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    if x.size == 0 or np.all(x == 0):
        return 0.0
    x = np.sort(x)
    n = len(x)
    idx = np.arange(1, n + 1)
    return float((2 * np.sum(idx * x)) / (n * np.sum(x)) - (n + 1) / n)


def survival_rounds(rec: dict) -> int | None:
    m = rec.get("metrics") or {}
    return m.get("survival_rounds")


def violation_rate(rec: dict) -> float | None:
    m = rec.get("metrics") or {}
    return m.get("violation_rate")


soft = [r for r in records if r.get("condition") == "soft"]
hard = [r for r in records if r.get("condition") == "hard"]
s_surv = [survival_rounds(r) for r in soft if survival_rounds(r) is not None]
h_surv = [survival_rounds(r) for r in hard if survival_rounds(r) is not None]
if len(s_surv) >= 2 and len(h_surv) >= 2:
    u, p = mannwhitneyu(s_surv, h_surv, alternative="two-sided")
    print("Mann-Whitney U survival:", u, "p=", p)
else:
    print("Not enough metric fields yet; populate metrics when runs are real.")